# APS360 Handwriting Recognition: Step 4 (EMNIST character-classifier baseline)

The proposal's second baseline: **cut each IAM word into characters with simple rules, then classify every character with a small CNN trained only on EMNIST** (handwritten single characters). It never sees IAM during training, and it can't use context between letters, which is exactly what the CRNN + CTC model adds. It runs on the **same IAM val and test words** as the CRNN and Tesseract (same labels, filters, writer-independent Aachen splits, same broken-image skipping) and is scored with the **same CER/WER code** (case-sensitive).

| Section | What happens |
|---|---|
| Setup | Drive, code, data (reuses `words_recovered.txt` from Step 1) |
| EMNIST | download (no sign-in; mirrors tried in turn), fix the transposed images, preview |
| Train the character CNN | ~0.45 M parameters, 6 epochs, checkpoints on Drive (re-run to resume) |
| Letter splitter | binarize, slant correction, connected components, projection-profile cuts, EMNIST-style 28x28 crops |
| Tune on validation | character width and the IAM character prior, chosen by **val** CER |
| Run | val + test (optional `N_TEST`), predictions saved to `outputs/report/` |
| Report outputs | classifier accuracy, baseline metrics, segmentation examples, failure grid, pipeline diagram, error analysis, and the **CRNN vs Tesseract vs EMNIST** table |

**Runtime: GPU (T4)** (Runtime → Change runtime type → T4 GPU): training takes a few minutes on a T4 but ~30+ min on a CPU. The splitter itself runs on the CPU.
**Time (estimate, not measured on Colab):** ~4–6 min setup (IAM unzip, same as the other steps), ~1 min EMNIST download, ~3–6 min training, ~1–2 min tuning, ~3–5 min for all val + test words: **about 15–20 min in total**.
**Run Step 1 first** (at least once) so `words_recovered.txt` exists. Steps 2 and 3 don't need to be re-run: this notebook rebuilds the comparison table with every model that has results.

## Mount Google Drive
Your Drive appears at `/content/drive/MyDrive`. Allow access when Colab asks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Paths
`PROJECT_DIR` is the Drive folder that holds `archive.zip`. If it has been moved, the cell searches your Drive for it.
Heavy files go on Colab's local disk (`/content`), which is far faster than Drive. Results are saved back to Drive.

In [ ]:
import os, subprocess
PROJECT_DIR = "/content/drive/MyDrive/University (Mech Eng)/Year 3/Fall/APS360 (Introduction to Machine Learning)/Project/Project Progress Report"
if not os.path.exists(os.path.join(PROJECT_DIR, "archive.zip")):
    print("archive.zip not found at PROJECT_DIR; searching Drive (can take a minute)...")
    hits = [h for h in subprocess.run(["find", "/content/drive/MyDrive", "-name", "archive.zip", "-size", "+500M"],
                                      capture_output=True, text=True).stdout.split("\n") if h]
    assert hits, "Could not find archive.zip anywhere in your Drive"
    PROJECT_DIR = os.path.dirname(hits[0])

ARCHIVE    = os.path.join(PROJECT_DIR, "archive.zip")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")       # results saved back to Drive
REPORT_DIR = os.path.join(OUTPUT_DIR, "report")         # figures/tables for the progress report
OFFICIAL_WORDS_TXT = os.path.join(PROJECT_DIR, "words.txt")            # full official file, if you add it
RECOVERED_WORDS_TXT = os.path.join(PROJECT_DIR, "words_recovered.txt") # made by the label-recovery step
CODE_DIR   = "/content/aps360_htr"                      # project code (this session only)
RAW_DIR    = "/content/iam_raw"                         # archive.zip unpacked here
DATA_ROOT  = "/content/iam"                             # tidy IAM layout (symlinks into RAW_DIR)
os.environ["HTR_DATA_ROOT"] = DATA_ROOT                 # src/config.py reads this
os.makedirs(REPORT_DIR, exist_ok=True)
print("PROJECT_DIR:", PROJECT_DIR)

## Write the project code
The baseline code is shown below as `%%writefile` cells. Everything else (data pipeline, metrics, report helpers) is identical to Steps 1-3 and comes from the embedded zip.

| File | What it does |
|---|---|
| `src/baselines/emnist.py` | EMNIST download (Hugging Face mirrors, then NIST) and checks, orientation fix, CNN, GPU training with checkpoints |
| `src/baselines/char_segment.py` | letter splitter (binarize, deslant, components, cuts, merges), punctuation rules, 28x28 normalization, decoding |
| `src/baselines/emnist_pipeline.py` | runs the splitter + CNN on IAM words, tuning on val, timing, skipping unreadable images |
| `src/report/emnist_report.py` | classifier table, metrics/tuning tables, segmentation figure, failure grid, pipeline diagram |
| `src/report/compare.py` | CRNN vs every baseline with results, on exactly the same words |

### The rest of the codebase (embedded zip)
Every step notebook is generated from one repo; this one embeds code **v16** (`aps360_htr_code_v16.zip` in your project folder).
This cell unpacks the 77 files that aren't shown below (see the other notebook or the zip to read them).
The long string is just that zip, base64-encoded.

In [ ]:
import base64, io, os, shutil, sys, zipfile
# Start from a clean copy every time: delete any older code left in /content/aps360_htr,
# and forget already-imported project modules so this session can't keep using stale code.
os.chdir("/content")
shutil.rmtree("/content/aps360_htr", ignore_errors=True)
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]
os.makedirs("/content/aps360_htr", exist_ok=True)
CODE_B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).extractall("/content/aps360_htr")
with open("/content/aps360_htr/CODE_VERSION", "w") as f:
    f.write("v16\n")
print("unpacked", len(zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).namelist()), "files | code v16")

In [ ]:
import os
for d in sorted({os.path.dirname(f) for f in ['src/baselines/__init__.py', 'src/baselines/emnist.py', 'src/baselines/char_segment.py', 'src/baselines/emnist_pipeline.py', 'src/report/emnist_report.py', 'src/report/compare.py']}):
    os.makedirs(f'/content/aps360_htr/{d}', exist_ok=True)
%cd /content/aps360_htr

### EMNIST data and the character CNN
Download with fallbacks, the transposition fix, the model and resumable training.

In [ ]:
%%writefile /content/aps360_htr/src/baselines/__init__.py
"""Baselines the CRNN is compared against (Tesseract OCR; EMNIST is future work)."""

In [ ]:
%%writefile /content/aps360_htr/src/baselines/emnist.py
"""EMNIST character classifier: data download, a small CNN, and resumable training.

Second baseline from the proposal: segment each IAM word into characters (see char_segment.py)
and classify every crop with a CNN trained only on EMNIST (no IAM training at all).

Split: **byclass** (62 classes: 0-9, A-Z, a-z). It is the only EMNIST split that keeps upper- and
lower-case letters apart, which our case-sensitive CER needs ('balanced' merges 15 lower-case letters
such as c/o/s into their capitals). byclass is unbalanced (many digits), so training uses mildly
re-weighted classes and recognition can use the IAM *training* character frequencies as a prior.

Data sources, tried in order, no sign-in needed (all serve the original NIST idx files):
  1. Hugging Face mirror  Royc30ne/emnist-byclass
  2. Hugging Face mirror  Heliosoph/EMNIST
  3. NIST original        biometrics.nist.gov gzip.zip (562 MB, all splits)
Each file is checked (gzip + idx header + item count) before it is used.

EMNIST images are stored **transposed** (a known quirk of the release). `orient()` checks the
mean image of the digit '1' (a vertical stroke when upright) and transposes if needed, so the
code is right whichever way a mirror stores them. Images are white ink on black.
"""
import gzip
import io
import json
import os
import shutil
import struct
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

CLASSES = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"   # byclass label order
FILES = {
    "train_x": "emnist-byclass-train-images-idx3-ubyte.gz",
    "train_y": "emnist-byclass-train-labels-idx1-ubyte.gz",
    "test_x": "emnist-byclass-test-images-idx3-ubyte.gz",
    "test_y": "emnist-byclass-test-labels-idx1-ubyte.gz",
}
COUNTS = {"train": 697_932, "test": 116_323}
HF = "https://huggingface.co/datasets/{repo}/resolve/main/{name}"
SOURCES = [
    ("Hugging Face mirror Royc30ne/emnist-byclass", "hf", "Royc30ne/emnist-byclass"),
    ("Hugging Face mirror Heliosoph/EMNIST", "hf", "Heliosoph/EMNIST"),
    ("NIST original gzip.zip (562 MB)", "nist", "https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip"),
]


# ------------------------------------------------------------------ download + reading

def read_idx(path):
    """Read a (gzipped) idx file into a uint8 numpy array."""
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rb") as f:
        data = f.read()
    zero, dtype, ndim = struct.unpack(">HBB", data[:4])
    if zero != 0 or dtype != 0x08:
        raise ValueError(f"{path}: not a uint8 idx file")
    shape = struct.unpack(">" + "I" * ndim, data[4:4 + 4 * ndim])
    arr = np.frombuffer(data, dtype=np.uint8, offset=4 + 4 * ndim)
    if arr.size != int(np.prod(shape)):
        raise ValueError(f"{path}: truncated ({arr.size} of {int(np.prod(shape))} values)")
    return arr.reshape(shape)


def _valid(path, expect_n=None):
    try:
        with gzip.open(path, "rb") as f:
            head = f.read(8)
        zero, dtype, ndim, n = struct.unpack(">HBBI", head)
        return zero == 0 and dtype == 8 and (expect_n is None or n == expect_n)
    except Exception:
        return False


def _expected(key):
    return COUNTS["train" if key.startswith("train") else "test"]


def _fetch(url, dest, timeout=60, retries=3):
    tmp = Path(str(dest) + ".part")
    for attempt in range(1, retries + 1):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "aps360-htr"}),
                                        timeout=timeout) as r, open(tmp, "wb") as f:
                shutil.copyfileobj(r, f, length=1 << 20)
            tmp.replace(dest)
            return True
        except Exception as e:
            print(f"    attempt {attempt}/{retries} failed: {type(e).__name__}: {str(e)[:120]}")
            time.sleep(2 * attempt)
    tmp.unlink(missing_ok=True)
    return False


def download(dest_dir, sources=SOURCES, verbose=True):
    """Make sure the four byclass files are in `dest_dir` (skips files already there and valid).
    Tries each source in turn; returns the name of the source used (or 'cached')."""
    dest = Path(dest_dir); dest.mkdir(parents=True, exist_ok=True)
    def missing():
        return [k for k, n in FILES.items() if not _valid(dest / n, _expected(k))]
    if not missing():
        return "cached"
    for label, kind, where in sources:
        if verbose:
            print(f"downloading EMNIST byclass from {label} ...")
        t = time.time()
        if kind == "hf":
            for k in missing():
                if not _fetch(HF.format(repo=where, name=FILES[k]), dest / FILES[k]) or not _valid(dest / FILES[k], _expected(k)):
                    print(f"  {FILES[k]}: failed from this source")
                    break
        elif kind == "nist":
            z = dest / "gzip.zip"
            if _fetch(where, z, timeout=120):
                try:
                    with zipfile.ZipFile(z) as zf:
                        for k in missing():
                            with zf.open("gzip/" + FILES[k]) as src, open(dest / FILES[k], "wb") as out:
                                shutil.copyfileobj(src, out)
                except Exception as e:
                    print(f"  could not read gzip.zip: {e}")
                z.unlink(missing_ok=True)
        if not missing():
            if verbose:
                print(f"  done in {time.time() - t:.0f}s")
            return label
    raise RuntimeError("Could not download EMNIST from any source (Hugging Face mirrors or NIST). "
                       "Check the internet connection and re-run; files already downloaded are kept.")


def orient(images, labels):
    """Return (upright images, was_transposed). Uses the mean image of digit '1' (label 1):
    upright, its ink is concentrated in a few *columns*; transposed, in a few *rows*."""
    ones = images[labels == 1][:5000].astype(np.float32)
    if len(ones) == 0:
        return images, False
    m = ones.mean(0)
    col, row = m.sum(0), m.sum(1)
    peaked = lambda p: p.max() / (p.mean() + 1e-6)
    if peaked(row) > peaked(col):
        return np.ascontiguousarray(images.transpose(0, 2, 1)), True
    return images, False


def load_emnist(dest_dir, verbose=True):
    """Download (if needed) and read EMNIST byclass. Returns dict with train_x/train_y/test_x/test_y
    (uint8, images N x 28 x 28 upright, white ink on black) plus 'source' and 'transposed'."""
    src = download(dest_dir, verbose=verbose)
    d = {k: read_idx(Path(dest_dir) / n) for k, n in FILES.items()}
    d["train_x"], tr = orient(d["train_x"], d["train_y"])
    d["test_x"], _ = orient(d["test_x"], d["test_y"])
    d["source"], d["transposed"] = src, tr
    if verbose:
        print(f"EMNIST byclass: {len(d['train_y']):,} train / {len(d['test_y']):,} test images, "
              f"{len(CLASSES)} classes | source: {src} | stored transposed: {tr} (fixed)")
    return d


# ------------------------------------------------------------------ model

class CharCNN(nn.Module):
    """Small VGG-style CNN for 28x28 characters (~0.45 M parameters)."""
    def __init__(self, num_classes=len(CLASSES), dropout=0.3):
        super().__init__()
        def block(cin, cout):
            return [nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
        self.features = nn.Sequential(*block(1, 32), *block(32, 32), nn.MaxPool2d(2),       # 14x14
                                      *block(32, 64), *block(64, 64), nn.MaxPool2d(2),      # 7x7
                                      *block(64, 128), nn.MaxPool2d(2))                     # 3x3
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(128 * 9, 256),
                                  nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(256, num_classes))

    def forward(self, x):            # x: (B, 1, 28, 28) in [0, 1], white ink on black
        return self.head(self.features(x))


def to_input(u8):
    """uint8 (N,28,28) array or tensor -> float (N,1,28,28) in [0,1]."""
    t = torch.as_tensor(u8)
    return t.float().div_(255).unsqueeze(1)


def augment(x, g=None):
    """Random slant/rotation/scale/shift and stroke thickness, on the GPU (x: B,1,28,28)."""
    B, dev = x.shape[0], x.device
    r = lambda lo, hi: torch.empty(B, device=dev).uniform_(lo, hi)
    ang, shear, sc = r(-0.17, 0.17), r(-0.3, 0.3), r(0.85, 1.12)
    tx, ty = r(-0.1, 0.1), r(-0.1, 0.1)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.stack([torch.stack([cos / sc, (-sin + shear) / sc, tx], 1),
                         torch.stack([sin / sc, cos / sc, ty], 1)], 1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, align_corners=False, padding_mode="zeros")
    k = torch.rand(B, device=dev)
    thick = F.max_pool2d(x, 3, 1, 1)                 # dilate: thicker strokes
    thin = -F.max_pool2d(-x, 3, 1, 1)                # erode: thinner strokes
    x = torch.where((k < 0.25).view(B, 1, 1, 1), thick, x)
    x = torch.where((k > 0.85).view(B, 1, 1, 1), torch.maximum(thin, 0.6 * x), x)
    return x


def class_weights(labels, power=0.3):
    """Mild inverse-frequency weights (freq^-power, mean 1) so rare lower-case letters count more."""
    cnt = np.bincount(labels, minlength=len(CLASSES)).astype(np.float64) + 1
    w = cnt ** -power
    return torch.tensor(w / (w * cnt).sum() * cnt.sum(), dtype=torch.float32)


@torch.no_grad()
def predict_logits(model, u8, device, batch=4096):
    model.eval()
    out = []
    for i in range(0, len(u8), batch):
        out.append(model(to_input(u8[i:i + batch]).to(device)).float().cpu())
    return torch.cat(out) if out else torch.zeros(0, len(CLASSES))


def accuracy_report(logits, labels):
    """Overall, per-group and case-insensitive accuracy on EMNIST."""
    pred = logits.argmax(1).numpy()
    y = np.asarray(labels)
    lower = np.array([c.lower() for c in CLASSES])
    groups = {"digits": y < 10, "upper": (y >= 10) & (y < 36), "lower": y >= 36}
    rep = {"accuracy": float((pred == y).mean()),
           "accuracy_case_insensitive": float((lower[pred] == lower[y]).mean()), "n": int(len(y))}
    for g, m in groups.items():
        rep[f"accuracy_{g}"] = float((pred[m] == y[m]).mean()) if m.any() else None
    return rep


def train(data, out_dir, epochs=6, batch_size=512, lr=2e-3, val_frac=0.05, train_limit=None, device=None,
          seed=0, verbose=True):
    """Train CharCNN on EMNIST byclass with checkpoints in `out_dir` (resumes from last.pt).
    Data are kept on the GPU as uint8, so an epoch takes well under a minute on a T4.
    Returns (model, info) where info has history, best_val_acc, minutes, test metrics."""
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rng = np.random.default_rng(seed)
    n = len(data["train_y"])
    perm = rng.permutation(n)
    n_val = int(n * val_frac)
    val_idx, tr_idx = perm[:n_val], perm[n_val:]
    if train_limit:
        tr_idx = tr_idx[:train_limit]
    xs = torch.as_tensor(data["train_x"][tr_idx]).to(device)
    ys = torch.as_tensor(data["train_y"][tr_idx].astype(np.int64)).to(device)
    xv, yv = data["train_x"][val_idx], data["train_y"][val_idx]

    model = CharCNN().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps_per_epoch = (len(tr_idx) + batch_size - 1) // batch_size
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * steps_per_epoch, pct_start=0.15)
    amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    lossf = nn.CrossEntropyLoss(weight=class_weights(data["train_y"][tr_idx]).to(device), label_smoothing=0.05)
    hist, start, best = [], 1, 0.0
    cfg = {"epochs": epochs, "batch_size": batch_size, "lr": lr, "val_frac": val_frac,
           "train_limit": train_limit, "seed": seed, "train_images": int(len(tr_idx))}
    last = out / "last.pt"
    if last.exists():
        ck = torch.load(last, map_location=device, weights_only=False)
        if ck.get("cfg") == cfg:
            model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
            hist, best, start = ck["hist"], ck["best"], ck["epoch"] + 1
            if verbose:
                print(f"resuming EMNIST training from epoch {ck['epoch']} (best val acc {best:.4f})")
        elif verbose:
            print("last.pt was trained with different settings; starting a new run (old files are overwritten)")
    g = torch.Generator(device=device); g.manual_seed(seed + start)
    for ep in range(start, epochs + 1):
        model.train(); t = time.time(); tot, cnt = 0.0, 0
        order = torch.randperm(len(ys), device=device, generator=g)
        for i in range(steps_per_epoch):
            idx = order[i * batch_size:(i + 1) * batch_size]
            x = augment(xs[idx].float().div_(255).unsqueeze(1))
            with torch.autocast(device.type, enabled=amp):
                loss = lossf(model(x), ys[idx])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(idx); cnt += len(idx)
        va = accuracy_report(predict_logits(model, xv, device), yv)
        row = {"epoch": ep, "train_loss": round(tot / cnt, 4), "val_acc": round(va["accuracy"], 4),
               "val_acc_case_insensitive": round(va["accuracy_case_insensitive"], 4), "secs": round(time.time() - t, 1)}
        hist.append(row)
        if va["accuracy"] > best:
            best = va["accuracy"]
            torch.save({"model": model.state_dict(), "epoch": ep, "val_acc": best, "classes": CLASSES}, out / "best.pt")
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "epoch": ep, "hist": hist, "best": best, "cfg": cfg}, last)
        if verbose:
            print(f"epoch {ep}/{epochs} | loss {row['train_loss']:.3f} | val acc {row['val_acc']:.4f} "
                  f"(case-insens. {row['val_acc_case_insensitive']:.4f}) | {row['secs']:.0f}s")
    del xs, ys
    best_ck = torch.load(out / "best.pt", map_location=device, weights_only=False)
    model.load_state_dict(best_ck["model"])
    test = accuracy_report(predict_logits(model, data["test_x"], device), data["test_y"])
    info = {"history": hist, "best_val_acc": best, "best_epoch": best_ck["epoch"], "cfg": cfg,
            "minutes": sum(h["secs"] for h in hist) / 60, "test": test,
            "parameters": sum(p.numel() for p in model.parameters()),
            "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU",
            "source": data.get("source"), "transposed": bool(data.get("transposed"))}
    (out / "emnist_classifier.json").write_text(json.dumps(info, indent=2))
    with open(out / "history.csv", "w") as f:
        f.write(",".join(hist[0]) + "\n")
        for h in hist:
            f.write(",".join(str(v) for v in h.values()) + "\n")
    if verbose:
        print(f"EMNIST test accuracy {test['accuracy']:.4f} (case-insensitive {test['accuracy_case_insensitive']:.4f})")
    return model, info


def load_classifier(path, device):
    ck = torch.load(path, map_location=device, weights_only=False)
    m = CharCNN().to(device); m.load_state_dict(ck["model"]); m.eval()
    return m

### The rule-based letter splitter
Every rule (and how punctuation EMNIST doesn't have is handled) is documented at the top of the file.

In [ ]:
%%writefile /content/aps360_htr/src/baselines/char_segment.py
"""Rule-based letter splitter for IAM word images + EMNIST-style normalization + word decoding.

Pipeline for one word image (all rules, no learning):
  1. binarize: light blur + Otsu threshold (ink = True); then **slant correction**: shear the word
     so its strokes are vertical (the shear that maximizes the sum of squared column ink counts),
     otherwise slanted letters overlap and vertical cuts slice through them;
  2. connected components (8-connected), then merge components that overlap horizontally
     (the dot of an i/j, broken strokes) into one group;
  3. estimate the pen's stroke width (2 x ink area / ink perimeter); the expected character width is
     `width_factor` x stroke width (on IAM training words, width / characters / stroke width has a
     median of ~7 and varies less between writers than the x-height); the core (x-)height band,
     from the horizontal ink profile, is used for the punctuation rules;
  4. small groups are punctuation candidates (see `_punct`); everything else is a letter group;
  5. a letter group wider than `split_ratio` expected widths is cut into n = round(width / expected
     width) pieces at the columns with the least ink (vertical projection profile) near the even cuts;
  6. pieces narrower than `min_piece` expected widths are merged into their narrower neighbour;
  7. each piece is normalized like EMNIST: cropped to its ink, stroke thickness adjusted, scaled so
     the long side is 24 px, centered in 28x28 (2-px border), white ink on black, slightly blurred.

Characters EMNIST doesn't have: EMNIST only knows 0-9, A-Z, a-z. Small marks are labelled by
position/shape rules instead: '.' (dot on the baseline), ',' (taller mark at the baseline),
"'" (mark above the letters), '"' (two marks above), '-' (flat mark in the middle), and for
single-mark word images also ':' ';' '!'. Other symbols (#&()*+/?) are never output, so they
always count as errors: this is a limitation of the baseline, not hidden.
Scoring is case-sensitive, exactly like the CRNN and Tesseract.
"""
from dataclasses import asdict, dataclass

import cv2
import numpy as np
from PIL import Image

from src.baselines.emnist import CLASSES


@dataclass
class SegParams:
    width_factor: float = 8.0    # expected character width = width_factor x stroke width
    split_ratio: float = 1.45    # only split groups wider than this many expected widths
    min_piece: float = 0.3       # merge pieces narrower than this many expected widths
    overlap_merge: float = 0.45  # merge components overlapping by this fraction of the narrower one
    noise: float = 0.012         # drop components with area < noise x core height^2 (and < 12 px)


def load_gray(path):
    """uint8 grayscale image, or None if the file is missing, empty or can't be decoded."""
    try:
        with Image.open(path) as im:
            im.load()
            a = np.asarray(im.convert("L"))
        return a if a.size and min(a.shape) > 0 else None
    except Exception:
        return None


def binarize(gray):
    g = cv2.GaussianBlur(gray, (3, 3), 0)
    if int(g.max()) - int(g.min()) < 30:          # blank / flat image: no ink
        return np.zeros(gray.shape, bool)
    _, m = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return m > 0


SHEARS = np.round(np.arange(-0.8, 0.81, 0.1), 2)


def _shear(img, s, fill):
    """x' = x + s * (y - H/2) (positive s straightens right-leaning writing)."""
    H, W = img.shape
    pad = int(np.ceil(abs(s) * H / 2)) + 1
    M = np.float32([[1, s, pad - s * H / 2], [0, 1, 0]])
    return cv2.warpAffine(img, M, (W + 2 * pad, H), flags=cv2.INTER_NEAREST, borderValue=fill)


def deslant(mask, shears=SHEARS):
    """Return (sheared mask, shear) with the most vertical strokes."""
    if not mask.any():
        return mask, 0.0
    m8 = mask.astype(np.uint8)
    best, bs = -1.0, 0.0
    for s in shears:
        col = _shear(m8, s, 0).sum(0).astype(np.float64)
        score = (col ** 2).sum()
        if score > best:
            best, bs = score, float(s)
    return (_shear(m8, bs, 0) > 0) if bs else mask, bs


def _core(mask):
    """(top, bottom, height) of the core band: the rows with at least half the peak row ink."""
    rows = mask.sum(1).astype(float)
    if rows.max() == 0:
        return 0, mask.shape[0], mask.shape[0]
    on = rows >= 0.5 * rows.max()
    best, cur, start, bs = 0, 0, 0, 0
    for i, v in enumerate(on):
        if v:
            if cur == 0:
                start = i
            cur += 1
            if cur > best:
                best, bs = cur, start
        else:
            cur = 0
    ink_rows = np.nonzero(rows)[0]
    h = max(best, 0.25 * (ink_rows[-1] - ink_rows[0] + 1), 4)
    return bs, bs + best, h


class Group:
    __slots__ = ("x0", "x1", "y0", "y1", "labels", "kind", "char", "area")

    def __init__(self, x0, x1, y0, y1, labels, area):
        self.x0, self.x1, self.y0, self.y1, self.labels, self.area = x0, x1, y0, y1, set(labels), area
        self.kind, self.char = "char", None

    @property
    def w(self):
        return self.x1 - self.x0

    @property
    def h(self):
        return self.y1 - self.y0

    def merge(self, o):
        self.x0, self.x1 = min(self.x0, o.x0), max(self.x1, o.x1)
        self.y0, self.y1 = min(self.y0, o.y0), max(self.y1, o.y1)
        self.labels |= o.labels; self.area += o.area


def _single_mark(groups, H, W):
    """Rules for a word image that is only punctuation (IAM crops are tight, so use absolute sizes;
    IAM is scanned at 300 dpi: a period is ~8 px, a lower-case letter ~30-40 px high)."""
    if W > 60 or H > 70:
        return None
    if len(groups) == 1:
        g = groups[0]
        if g.h <= 24 and g.w <= 26 and 0.5 <= g.h / max(g.w, 1) <= 2.0:
            return "."
        if g.h <= 16 and g.w >= 1.8 * g.h:
            return "-"
        if g.h <= 45 and g.w <= 30 and g.h >= 1.4 * g.w:
            return ","
        return None
    if len(groups) == 2:
        a, b = sorted(groups, key=lambda g: g.y0)
        stacked = min(a.x1, b.x1) - max(a.x0, b.x0) > 0 or abs((a.x0 + a.x1) - (b.x0 + b.x1)) < max(a.w, b.w)
        side = not stacked and abs(a.y0 - b.y0) < 0.5 * max(a.h, b.h)
        if side and max(a.h, b.h) <= 45:
            return '"'
        if stacked and b.y0 >= a.y1 - 2:
            small = lambda g: g.h <= 24 and g.w <= 26
            if small(a) and small(b):
                return ":"
            if small(a) and b.h >= 1.4 * b.w and b.h <= 45:
                return ";"
            if small(b) and a.h >= 2 * a.w:
                return "!"
    return None


def _punct(g, core_top, core_bot, xh):
    """Label a small group inside a longer word by its size and position relative to the core band."""
    if g.h > 0.6 * xh or g.w > 0.8 * xh:
        if g.h <= 0.35 * xh and g.w >= 1.8 * g.h and core_top - 0.1 * xh <= (g.y0 + g.y1) / 2 <= core_bot:
            return "-"
        return None
    cy = (g.y0 + g.y1) / 2
    if g.y1 <= core_top + 0.2 * xh:
        return "'"
    if cy >= core_bot - 0.25 * xh:
        return "." if g.h <= 1.3 * max(g.w, 1) else ","
    if g.w >= 1.8 * g.h:
        return "-"
    return None


def segment(gray, p: SegParams = SegParams()):
    """Split a word image into character pieces. Returns (pieces, info); each piece is a dict
    with x0, x1, y0, y1, kind ('char' or 'punct'), char (for punct) and mask (bool crop)."""
    mask = binarize(gray)
    mask, shear = deslant(mask)
    H, W = mask.shape
    if not mask.any():
        return [], {"core": (0, H, H), "shear": 0.0}
    n, lab, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    ct, cb, xh = _core(mask)
    min_area = min(12, p.noise * xh * xh)
    groups = [Group(s[0], s[0] + s[2], s[1], s[1] + s[3], [i], s[4])
              for i, s in enumerate(stats) if i > 0 and s[4] >= max(min_area, 2)]
    if not groups:
        return [], {"core": (ct, cb, xh), "shear": shear}
    keep = np.isin(lab, [next(iter(g.labels)) for g in groups])
    ct, cb, xh = _core(keep)
    # 2. merge horizontally overlapping components
    groups.sort(key=lambda g: g.x0)
    changed = True
    while changed:
        changed, out = False, []
        for g in groups:
            if out:
                o = out[-1]
                ov = min(o.x1, g.x1) - max(o.x0, g.x0)
                if ov > 0 and ov >= p.overlap_merge * min(o.w, g.w):
                    o.merge(g); changed = True; continue
            out.append(g)
        groups = sorted(out, key=lambda g: g.x0)
    # 4. punctuation
    whole = _single_mark(groups, H, W)
    if whole is not None:
        return [{"x0": 0, "x1": W, "y0": 0, "y1": H, "kind": "punct", "char": whole, "mask": keep}], \
               {"core": (ct, cb, xh), "shear": shear}
    if len(groups) > 1:
        for g in groups:
            c = _punct(g, ct, cb, xh)
            if c is not None:
                g.kind, g.char = "punct", c
        letters = [g for g in groups if g.kind == "char"]
        if not letters:                       # everything looked like punctuation: treat as letters
            for g in groups:
                g.kind, g.char = "char", None
        # two apostrophes side by side -> one double quote
        merged = []
        for g in groups:
            if merged and g.char == "'" and merged[-1].char == "'" and g.x0 - merged[-1].x1 < 0.5 * xh:
                merged[-1].merge(g); merged[-1].char = '"'; continue
            merged.append(g)
        groups = merged
    # 5. split wide letter groups at projection-profile minima
    cw = max(p.width_factor * _stroke_width(keep), 3)
    pieces = []
    for g in groups:
        gm = np.isin(lab[:, g.x0:g.x1], list(g.labels))
        if g.kind != "char":
            pieces.append({"x0": g.x0, "x1": g.x1, "kind": "punct", "char": g.char, "gm": gm, "off": g.x0})
            continue
        k = int(round(g.w / cw))
        cuts = []
        if k >= 2 and g.w > p.split_ratio * cw:
            prof = gm.sum(0).astype(float)
            step = g.w / k
            for j in range(1, k):
                t = j * step
                lo, hi = int(max(1, t - 0.4 * step)), int(min(g.w - 1, t + 0.4 * step))
                if hi <= lo:
                    continue
                cols = np.arange(lo, hi)
                cost = prof[lo:hi] + 0.15 * prof.max() * np.abs(cols - t) / step
                cuts.append(int(cols[np.argmin(cost)]))
        edges = [0] + sorted(set(cuts)) + [g.w]
        for a, b in zip(edges[:-1], edges[1:]):
            if b > a:
                pieces.append({"x0": g.x0 + a, "x1": g.x0 + b, "kind": "char", "char": None,
                               "gm": gm[:, a:b], "off": g.x0 + a})
    # 6. merge thin letter pieces into the narrower neighbour
    def width(q): return q["x1"] - q["x0"]
    i = 0
    while i < len(pieces):
        q = pieces[i]
        if q["kind"] == "char" and width(q) < p.min_piece * cw and len(pieces) > 1:
            nb = [j for j in (i - 1, i + 1) if 0 <= j < len(pieces) and pieces[j]["kind"] == "char"]
            if nb:
                j = min(nb, key=lambda j: width(pieces[j]))
                a, b = sorted((i, j))
                L, R = pieces[a], pieces[b]
                x0, x1 = min(L["x0"], R["x0"]), max(L["x1"], R["x1"])
                gm = np.zeros((H, x1 - x0), bool)
                for s in (L, R):
                    gm[:, s["x0"] - x0:s["x1"] - x0] |= s["gm"]
                pieces[a] = {"x0": x0, "x1": x1, "kind": "char", "char": None, "gm": gm, "off": x0}
                del pieces[b]
                i = max(a - 1, 0)
                continue
        i += 1
    out = []
    for q in pieces:
        rows = np.nonzero(q["gm"].any(1))[0]
        if len(rows) == 0:
            continue
        y0, y1 = int(rows[0]), int(rows[-1]) + 1
        out.append({"x0": int(q["x0"]), "x1": int(q["x1"]), "y0": y0, "y1": y1, "kind": q["kind"],
                    "char": q["char"], "mask": q["gm"][y0:y1]})
    return out, {"core": (ct, cb, xh), "shear": shear}


def _stroke_width(m):
    """Mean stroke thickness in px (2 x area / perimeter)."""
    cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
    per = sum(cv2.arcLength(c, True) for c in cs)
    return 2.0 * m.sum() / max(per, 1.0)


def to_emnist(mask, size=28, box=24, target_stroke=2.8):
    """One character's ink mask -> 28x28 uint8 like EMNIST (white on black, long side 24 px,
    bounding box centered, stroke thickness matched to EMNIST, light blur)."""
    ys, xs = np.nonzero(mask)
    if len(ys) == 0:
        return np.zeros((size, size), np.uint8)
    m = mask[ys.min():ys.max() + 1, xs.min():xs.max() + 1].astype(np.uint8)
    h, w = m.shape
    scale = box / max(h, w)
    sw = _stroke_width(m) * scale            # stroke width it would have at 28x28
    pad = 4
    m = np.pad(m, pad)
    if sw < 0.8 * target_stroke:
        k = int(np.clip(round((target_stroke - sw) / scale), 1, 15))
        m = cv2.dilate(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k + 1, k + 1)))
    elif sw > 1.6 * target_stroke:
        k = int(np.clip(round((sw - target_stroke) / scale / 2), 1, 15))
        er = cv2.erode(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k + 1, k + 1)))
        m = er if er.any() else m
    ys, xs = np.nonzero(m)
    m = m[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = m.shape
    scale = box / max(h, w)
    nh, nw = max(1, round(h * scale)), max(1, round(w * scale))
    r = cv2.resize(m.astype(np.float32) * 255, (nw, nh), interpolation=cv2.INTER_AREA)
    out = np.zeros((size, size), np.float32)
    y, x = (size - nh) // 2, (size - nw) // 2
    out[y:y + nh, x:x + nw] = r
    out = cv2.GaussianBlur(out, (3, 3), 0.7)
    return np.clip(out * (255.0 / max(out.max(), 1)), 0, 255).astype(np.uint8)


def deslanted_gray(gray, shear):
    """The grayscale word with the same shear as the mask (for figures)."""
    return _shear(gray, shear, 255) if shear else gray


def segment_word(gray, p: SegParams = SegParams(), return_info=False):
    """-> (pieces, crops) where crops is (n_char_pieces, 28, 28) uint8 for the 'char' pieces
    (piece coordinates are in the slant-corrected image; see deslanted_gray)."""
    pieces, info = segment(gray, p)
    crops = [to_emnist(q["mask"]) for q in pieces if q["kind"] == "char"]
    crops = np.stack(crops) if crops else np.zeros((0, 28, 28), np.uint8)
    return (pieces, crops, info) if return_info else (pieces, crops)


# ------------------------------------------------------------------ decoding with a character prior

def char_prior(train_texts, emnist_train_labels, power=0.3):
    """log p_IAM(c) - log p_EMNIST_eff(c) for the 62 EMNIST classes. p_IAM comes from the IAM
    *training* labels only; p_EMNIST_eff is the class frequency the classifier effectively saw
    (counts^(1-power) because of the class weights used in training)."""
    idx = {c: i for i, c in enumerate(CLASSES)}
    iam = np.ones(len(CLASSES))
    for t in train_texts:
        for c in t:
            if c in idx:
                iam[idx[c]] += 1
    em = (np.bincount(emnist_train_labels, minlength=len(CLASSES)).astype(float) + 1) ** (1 - power)
    return np.log(iam / iam.sum()) - np.log(em / em.sum())


def decode(pieces, logp, prior=None, alpha=0.0):
    """Join the word: punctuation pieces keep their rule label, letter pieces take the best class
    of log p(class | crop) + alpha * prior."""
    s = logp if (prior is None or alpha == 0) else logp + alpha * prior[None, :]
    best = s.argmax(1) if len(s) else []
    out, k = [], 0
    for q in pieces:
        if q["kind"] == "punct":
            out.append(q["char"])
        else:
            out.append(CLASSES[int(best[k])]); k += 1
    return "".join(out)


def params_dict(p):
    return asdict(p)

In [ ]:
%%writefile /content/aps360_htr/src/baselines/emnist_pipeline.py
"""EMNIST baseline end to end on IAM words: segment -> normalize -> classify -> join -> score.

Settings that are tuned (character width factor, and whether to use the IAM character prior) are
chosen on **validation** words only; the test split is run once with the chosen settings.
Unreadable images are skipped and listed, exactly like the CRNN and Tesseract runs.
"""
import time
from dataclasses import replace

import numpy as np
import torch

from src.baselines import char_segment as cs
from src.baselines.emnist import CLASSES, predict_logits
from src.utils.metrics import cer, wer


def segment_items(items, params=cs.SegParams(), progress=True, quiet=False):
    """Segment every word. Returns (segs, skipped_ids, seconds); segs keep pieces (without masks)
    and the 28x28 crops."""
    segs, skipped, t0 = [], [], time.time()
    step = max(1, len(items) // 5)
    for i, it in enumerate(items):
        g = cs.load_gray(it.path)
        if g is None:
            skipped.append(it.word_id); continue
        pieces, crops, info = cs.segment_word(g, params, return_info=True)
        segs.append({"item": it, "pieces": [{k: v for k, v in q.items() if k != "mask"} for q in pieces],
                     "crops": crops, "shear": info.get("shear", 0.0)})
        if progress and (i + 1) % step == 0:
            el = time.time() - t0
            print(f"  segmented {i + 1:,}/{len(items):,} words, {el:.0f}s elapsed, ~{el / (i + 1) * (len(items) - i - 1):.0f}s left")
    if skipped and not quiet:
        print(f"  skipped {len(skipped)} unreadable image(s): {', '.join(skipped[:10])}")
    return segs, skipped, time.time() - t0


def classify(segs, model, device):
    """log-probabilities for every crop, split back per word. Returns (list of arrays, seconds)."""
    t0 = time.time()
    allc = [s["crops"] for s in segs]
    sizes = [len(c) for c in allc]
    flat = np.concatenate(allc) if sum(sizes) else np.zeros((0, 28, 28), np.uint8)
    lp = torch.log_softmax(predict_logits(model, flat, device), 1).numpy() if len(flat) else \
        np.zeros((0, len(CLASSES)), np.float32)
    out, k = [], 0
    for n in sizes:
        out.append(lp[k:k + n]); k += n
    return out, time.time() - t0


def decode_all(segs, logps, prior=None, alpha=0.0):
    rows = []
    for s, lp in zip(segs, logps):
        it = s["item"]
        h = cs.decode(s["pieces"], lp, prior, alpha)
        rows.append({"word_id": it.word_id, "writer": it.writer or "", "ref": it.text, "hyp": h,
                     "correct": str(int(h == it.text)), "pieces": len(s["pieces"])})
    return rows


def scores(rows):
    refs, hyps = [r["ref"] for r in rows], [r["hyp"] for r in rows]
    return {"cer": cer(refs, hyps), "wer": wer(refs, hyps),
            "cer_case_insensitive": cer([r.lower() for r in refs], [h.lower() for h in hyps]),
            "wer_case_insensitive": wer([r.lower() for r in refs], [h.lower() for h in hyps]),
            "words": len(rows)}


def tune(val_items, model, device, prior, width_factors=(6.0, 7.0, 8.0, 9.0, 10.0, 11.0), alphas=(0.0, 1.0),
         base=cs.SegParams()):
    """Grid over the width factor and prior on/off, scored by validation CER. Returns
    (best_params, best_alpha, table rows sorted best first)."""
    table = []
    for wf in width_factors:
        p = replace(base, width_factor=wf)
        segs, _, _ = segment_items(val_items, p, progress=False, quiet=True)
        logps, _ = classify(segs, model, device)
        for a in alphas:
            s = scores(decode_all(segs, logps, prior, a))
            table.append({"width_factor": wf, "prior": a, **s})
            print(f"  width factor {wf:.1f} x stroke | prior {'on ' if a else 'off'} | val CER {s['cer']:.4f} | WER {s['wer']:.4f}")
    table.sort(key=lambda r: (r["cer"], r["wer"]))
    best = table[0]
    return replace(base, width_factor=best["width_factor"]), best["prior"], table


def run(items, model, device, params, prior=None, alpha=0.0, progress=True):
    """Recognize `items`. Returns (rows, info) with timing (segmentation on CPU, classification
    batched on `device`) and the skipped word IDs."""
    segs, skipped, t_seg = segment_items(items, params, progress)
    logps, t_cls = classify(segs, model, device)
    rows = decode_all(segs, logps, prior, alpha)
    n = max(len(rows), 1)
    info = {"skipped": skipped, "seg_ms_per_word": 1000 * t_seg / n, "cls_ms_per_word": 1000 * t_cls / n,
            "ms_per_word": 1000 * (t_seg + t_cls) / n, "crops": int(sum(len(s["crops"]) for s in segs)),
            "segs": segs, "logps": logps}
    return rows, info


FIELDS = ["word_id", "writer", "ref", "hyp", "correct", "pieces"]


def save_rows(rows, path):
    import csv
    import os
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows({k: r[k] for k in FIELDS} for r in rows)
    return str(path)

### Report outputs and the three-way comparison

In [ ]:
%%writefile /content/aps360_htr/src/report/emnist_report.py
"""Step 4: report figures, tables and numbers for the EMNIST character-classifier baseline."""
import json
import random

from src.baselines import char_segment as cs
from src.baselines.emnist import CLASSES
from src.report import figures, tables
from src.report.predict import original_image

pct = lambda x: f"{100 * x:.2f}"


def save_classifier(rep, info):
    """EMNIST classifier on its own: test accuracy (overall / per group / case-insensitive), size, training."""
    t = info["test"]
    rows = [{"Item": "Training data", "Value": f"EMNIST byclass ({len(CLASSES)} classes: 0-9, A-Z, a-z), "
                                               f"{info['cfg']['train_images']:,} training images"},
            {"Item": "Test accuracy", "Value": f"{pct(t['accuracy'])}% ({t['n']:,} EMNIST test images)"},
            {"Item": "Test accuracy, case-insensitive", "Value": f"{pct(t['accuracy_case_insensitive'])}%"},
            {"Item": "Digits / upper / lower", "Value": " / ".join(f"{pct(t['accuracy_' + g])}%" for g in ("digits", "upper", "lower"))},
            {"Item": "Parameters", "Value": f"{info['parameters']:,}"},
            {"Item": "Training", "Value": f"{len(info['history'])} epochs (best {info['best_epoch']}), "
                                          f"{info['minutes']:.1f} min on {info['device']}"}]
    paths = tables.write_table(rows, ["Item", "Value"], rep.dir, "table_emnist_classifier",
                               "The EMNIST character classifier on its own (EMNIST byclass test set). Upper/lower-case "
                               "pairs such as o/O, s/S and c/C look almost the same, which is why byclass accuracy "
                               "is capped near 88% and the case-insensitive accuracy is higher.",
                               "tab:emnist_classifier", col_format="lp{3.45in}")
    rep.add(paths, "EMNIST character classifier: test accuracy overall, case-insensitive and per group, size and training time.")
    return paths


def save_metrics(rep, preds, infos, full_sizes, tuning, settings, device, classifier):
    """Word-level CER/WER on IAM val/test (same words and code as the CRNN), plus the val tuning table."""
    from src.baselines.emnist_pipeline import scores
    nums = {"settings": settings, "device": device, "classifier": {k: v for k, v in classifier.items() if k != "history"}}
    rows = []
    for split, p in preds.items():
        s = scores(p)
        alnum = [r for r in p if r["ref"].isalnum()]
        nums[split] = {**s, "of_words": full_sizes[split], "skipped": infos[split]["skipped"],
                       "ms_per_word": infos[split]["ms_per_word"], "seg_ms_per_word": infos[split]["seg_ms_per_word"],
                       "empty_outputs": sum(1 for r in p if not r["hyp"]),
                       "cer_letters_digits_only_words": scores(alnum)["cer"] if alnum else None}
        rows.append({"Split": split, "Words": f"{len(p):,} of {full_sizes[split]:,}", "CER (%)": pct(s["cer"]),
                     "WER (%)": pct(s["wer"]), "CER, no case (%)": pct(s["cer_case_insensitive"]),
                     "ms/word": f"{infos[split]['ms_per_word']:.1f}"})
    rep.numbers("emnist", nums)
    (rep.dir / "emnist_metrics.json").write_text(json.dumps(nums, indent=2))
    rep.add([str(rep.dir / "emnist_metrics.json")], "EMNIST baseline: val/test CER, WER (also case-insensitive), speed, settings.")
    paths = tables.write_table(rows, list(rows[0]), rep.dir, "table_emnist_metrics",
                               f"EMNIST character-classifier baseline (rule-based letter splitter + CNN trained only on "
                               f"EMNIST) on the IAM validation and test words. CER/WER are case-sensitive, as for the other "
                               f"models; 'no case' ignores upper/lower case. ms/word = segmentation on the CPU + "
                               f"classification on {device}.", "tab:emnist_metrics")
    rep.add(paths, "Baseline metrics: EMNIST baseline CER/WER on val and test (case-sensitive like the other models, "
                   "plus case-insensitive CER) and time per word.")
    trows = [{"Width factor": f"{r['width_factor']:.1f}", "IAM prior": "on" if r["prior"] else "off",
              "Val CER (%)": pct(r["cer"]), "Val WER (%)": pct(r["wer"]), "_bold": i == 0} for i, r in enumerate(tuning)]
    p2 = tables.write_table(trows, ["Width factor", "IAM prior", "Val CER (%)", "Val WER (%)"], rep.dir,
                            "table_emnist_tuning",
                            "EMNIST baseline settings chosen on validation words (best first, chosen in bold): the "
                            "expected character width as a multiple of the core (x-)height, and whether the classifier's "
                            "scores are re-weighted by IAM training-set character frequencies.", "tab:emnist_tuning")
    rep.add(p2, "EMNIST baseline tuning on validation words: splitter width factor and the IAM character prior (chosen row in bold).")
    return nums


def segmentation_examples(ds, segs, logps, prior, alpha, n=4, seed=0):
    """Pick `n` random test words (not cherry-picked) and collect what the figure needs."""
    idx = random.Random(seed).sample(range(len(segs)), min(n, len(segs)))
    out = []
    for i in idx:
        s, lp = segs[i], logps[i]
        it = s["item"]
        hyp = cs.decode(s["pieces"], lp, prior, alpha)
        sc = lp if (prior is None or not alpha) else lp + alpha * prior[None, :]
        letters = iter(CLASSES[int(k)] for k in (sc.argmax(1) if len(sc) else []))
        preds = [q["char"] if q["kind"] == "punct" else next(letters) for q in s["pieces"]]
        out.append({"gray": cs.deslanted_gray(original_image(it), s.get("shear", 0.0)), "pieces": s["pieces"], "crops": s["crops"], "preds": preds,
                    "ref": it.text, "hyp": hyp})
    return out


def save_segmentation_figure(rep, examples):
    paths = figures.plot_segmentation_examples(examples, rep.dir)
    rep.add(paths, f"EMNIST baseline on {len(examples)} randomly chosen test words (shown after slant correction): the splitter's character boxes "
                   "(red = classified crop, blue = punctuation rule), the normalized 28x28 crops and the predicted characters.")
    return paths


def save_failure_grid(rep, ds, preds, n=12, seed=0):
    wrong = [r for r in preds if r["ref"] != r["hyp"]]
    rows = random.Random(seed).sample(wrong, min(n, len(wrong)))
    if not rows:
        print("EMNIST baseline got every word right: no failure grid"); return None
    by_id = {it.word_id: it for it in ds.items}
    imgs = [original_image(by_id[r["word_id"]]) for r in rows]
    paths = figures.plot_prediction_grid(imgs, [r["ref"] for r in rows], [r["hyp"] for r in rows], rep.dir,
                                         "fig_emnist_failures")
    rep.add(paths, f"Typical EMNIST-baseline failures: {len(rows)} randomly chosen wrong test predictions (red).")
    return paths


def save_pipeline(rep):
    paths = figures.draw_emnist_pipeline(rep.dir)
    rep.add(paths, "EMNIST baseline pipeline: binarize, connected components, rule-based splitting/merging, "
                   "EMNIST-style normalization, a CNN trained only on EMNIST, then the characters are joined "
                   "and scored with the same CER/WER code as the CRNN.")
    return paths

In [ ]:
%%writefile /content/aps360_htr/src/report/compare.py
"""CRNN vs the baselines (Tesseract, EMNIST character classifier) on exactly the same words.

Needs crnn_metrics.json + crnn_<split>_predictions.csv (Step 2) and, for each baseline that has run,
<name>_metrics.json + <name>_<split>_predictions.csv (Step 3 = tesseract, Step 4 = emnist) in the
report folder. Whichever notebook runs last builds the table with every model available.
All models are scored on the words they **all** have (matched by word ID): if a baseline was run on
a random subset of the test set, everyone is scored on that subset, so the comparison is like-for-like.
"""
import json

from src.report import errors, figures, tables
from src.report.predict import load_predictions
from src.utils.metrics import cer, wer

pct = lambda x: f"{100 * x:.2f}"

SYSTEMS = [("CRNN (ours)", "crnn"), ("Tesseract (LSTM)", "tesseract"), ("EMNIST CNN + splitter", "emnist")]
BASELINES = SYSTEMS[1:]


def _scores(rows):
    refs, hyps = [r["ref"] for r in rows], [r["hyp"] for r in rows]
    return {"cer": cer(refs, hyps), "wer": wer(refs, hyps), "words": len(rows)}


def available(d):
    """{prefix: metrics dict} for every system whose metrics + test predictions exist."""
    out = {}
    for _, p in SYSTEMS:
        if (d / f"{p}_metrics.json").exists() and (d / f"{p}_test_predictions.csv").exists():
            out[p] = json.loads((d / f"{p}_metrics.json").read_text())
    return out


def _speed(prefix, m):
    if prefix == "crnn":
        return m.get("test", {}).get("ms_per_word_batched")
    if prefix == "tesseract":
        return m.get("test", {}).get("ms_per_word_cpu_core")
    return m.get("test", {}).get("ms_per_word")


def save_comparison(rep):
    d = rep.dir
    have = available(d)
    if "crnn" not in have or len(have) < 2:
        missing = ["Step 2 (CRNN)"] if "crnn" not in have else []
        if len(have) - ("crnn" in have) < 1:
            missing.append("a baseline (Step 3 Tesseract or Step 4 EMNIST)")
        print("Comparison table not made yet; missing results from:", " and ".join(missing))
        print("It is created automatically by whichever of Steps 2-4 finishes last.")
        return None
    systems = [(n, p) for n, p in SYSTEMS if p in have]
    per_split = {}
    for split in ("val", "test"):
        preds = {p: load_predictions(d / f"{p}_{split}_predictions.csv") for _, p in systems
                 if (d / f"{p}_{split}_predictions.csv").exists()}
        if "crnn" not in preds or len(preds) < 2:
            continue
        common = set.intersection(*({r["word_id"] for r in rows} for rows in preds.values()))
        rows = {p: sorted((r for r in v if r["word_id"] in common), key=lambda r: r["word_id"]) for p, v in preds.items()}
        per_split[split] = {p: _scores(v) for p, v in rows.items()}
        per_split[split]["_rows"] = rows
    params = {"crnn": rep.numbers().get("model", {}).get("parameters"),
              "emnist": have.get("emnist", {}).get("classifier", {}).get("parameters")}
    rows, nums = [], {}
    for name, p in systems:
        row = {"Model": name}
        for split in ("val", "test"):
            s = per_split.get(split, {}).get(p)
            row[f"{split.capitalize()} CER"] = pct(s["cer"]) if s else "--"
            row[f"{split.capitalize()} WER"] = pct(s["wer"]) if s else "--"
        ms = _speed(p, have[p])
        row["ms/word"] = "" if ms is None else (f"{ms:.2f}" if ms < 10 else f"{ms:.0f}")
        row["Params"] = f"{params[p] / 1e6:.2f}M" if params.get(p) else "--"
        rows.append(row)
        nums[name] = {s: per_split[s][p] for s in per_split if p in per_split[s]}
        nums[name]["ms_per_word"] = ms
    n_test = per_split["test"]["crnn"]["words"]
    full = have["crnn"].get("test", {}).get("words")
    note = None
    if full and n_test < full:
        note = (f"All models scored on the same {n_test:,} test words (the words every model was run on); "
                f"CRNN on all {full:,} test words: CER {pct(have['crnn']['test']['cer'])}%, "
                f"WER {pct(have['crnn']['test']['wer'])}%.")
    speed = {"crnn": f"CRNN batched (128) on {have['crnn'].get('device')}",
             "tesseract": "Tesseract per word on one CPU core",
             "emnist": f"EMNIST baseline: segmentation on the CPU + batched classification on "
                       f"{have.get('emnist', {}).get('device', 'GPU')}"}
    pre = {"tesseract": "Tesseract is pretrained on printed text", "emnist": "the EMNIST CNN is trained only on EMNIST characters"}
    base_txt = "; ".join(pre[p] for _, p in systems if p in pre)
    cap = (f"CRNN vs. the baselines on the same IAM words ({n_test:,} test words from writers unseen in training). "
           f"CER and WER in %, case-sensitive, lower is better. Speed: " + "; ".join(speed[p] for _, p in systems) +
           f". Neither baseline is trained on IAM ({base_txt}).")
    paths = tables.write_table(rows, list(rows[0]), d, "table_comparison", cap, "tab:comparison", note=note)
    rep.add(paths, "Main quantitative result: CRNN vs the baselines (" + ", ".join(n for n, p in systems if p != "crnn")
            + ") CER/WER on the same val/test words, speed and size.")
    sys_rows = per_split["test"]["_rows"]
    lens = {n: errors.cer_by_length([r["ref"] for r in sys_rows[p]], [r["hyp"] for r in sys_rows[p]])
            for n, p in systems if p in sys_rows}
    p2 = figures.plot_cer_by_length(lens, d, "fig_cer_by_length_comparison",
                                    title="CER by word length: " + " vs ".join(n.split(" (")[0] for n in lens)
                                    + " (same test words)")
    rep.add(p2, "CER by word length for " + ", ".join(lens) + " on the same test words.")
    nums["note"] = note
    nums["models"] = [n for n, _ in systems]
    nums["test_words"] = n_test
    rep.numbers("comparison", nums)
    return rows

## Install missing packages
Colab already has PyTorch, OpenCV, NumPy, Pillow, matplotlib, pyarrow and pytest, so this only installs what's missing.
`editdistance` (faster CER/WER) is optional; the code falls back to pure Python without it.

In [ ]:
import importlib.util, subprocess, sys
need = {"torch": "torch", "torchvision": "torchvision", "cv2": "opencv-python", "numpy": "numpy",
        "PIL": "pillow", "matplotlib": "matplotlib", "tqdm": "tqdm", "pytest": "pytest",
        "pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub",
        "pytesseract": "pytesseract", "editdistance": "editdistance"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
print("missing:", missing or "nothing")
for pkg in missing:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    note = " (optional: CER/WER use the pure-Python fallback)" if pkg == "editdistance" else ""
    print(f"  {pkg}: {'installed' if r.returncode == 0 else 'install failed' + note}")

## Unpack archive.zip (about 1.2 GB)
Copied to Colab's disk and unzipped there, which takes a few minutes. Re-running in the same session skips it.

In [ ]:
import os, shutil, time
marker = os.path.join(RAW_DIR, ".unzipped")
if os.path.exists(marker):
    print("Already unzipped this session:", RAW_DIR)
else:
    t = time.time()
    shutil.copy(ARCHIVE, "/content/archive.zip")
    print(f"copied in {time.time()-t:.0f}s"); t = time.time()
    !unzip -q -o /content/archive.zip -d {RAW_DIR}
    os.remove("/content/archive.zip")
    open(marker, "w").close()
    print(f"unzipped in {time.time()-t:.0f}s")
!ls {RAW_DIR}

## Find words.txt and the images
`setup_iam.py` finds the word images and a `words.txt` in any download layout (official tgz, Kaggle `iam_words/`, or a flat folder) and links them into `/content/iam`.

**Which labels get used** (first one that exists wins):
1. `words.txt` in your project folder: the full official file, if you download it from IAM;
2. `words_recovered.txt` in your project folder, made by the label-recovery step below;
3. the `words.txt` inside `archive.zip`. The Kaggle copy is **cut off at e07-072**, so only ~39% of IAM has labels.

In [ ]:
def pick_words_txt():
    for p in (OFFICIAL_WORDS_TXT, RECOVERED_WORDS_TXT):
        if os.path.exists(p):
            return p
    return None

def run_setup():
    wt = pick_words_txt()
    print("labels from:", wt or "the words.txt inside archive.zip")
    extra = f'--words-txt "{wt}"' if wt else ""
    !python scripts/setup_iam.py --raw {RAW_DIR} --out {DATA_ROOT} {extra}

run_setup()

## Recover the missing labels (only needed once)
**Why:** the Kaggle `words.txt` stops part-way through line `e07-072-09-`, but `archive.zip` has the images for all of IAM. Without the missing lines, about 60% of the images can't be used.

**How:** every original IAM PNG has its form's XML stored *inside the file* (a PNG text chunk). That XML gives the word's ID, its text, its tag, its bounding box and its line's segmentation status (ok/err), which are exactly the fields of a `words.txt` line. `recover_labels.py`:
1. first tries the XML inside **your own** images (fast, no download);
2. if they've been stripped, it downloads the Hugging Face copy of IAM ([priyank-m/IAM_words_text_recognition](https://huggingface.co/datasets/priyank-m/IAM_words_text_recognition), ~1.2 GB), whose PNGs still carry the XML, and matches **by word ID**. Images whose ID can't be found fall back to a pixel-hash match.

**Checks it prints, for the report:**
- *rebuilt vs original*: for words that have both, the rebuilt line must equal the original Kaggle line (expect ~100%);
- *pixel check*: the HF image is pixel-identical to yours for a sample of recovered words;
- matched / unmatched / ambiguous counts. Below 90%, nothing is written and training uses the labelled subset.

The result is saved to your project folder as `words_recovered.txt` (original lines first, then a `#--- recovered entries below ---` section), so later sessions and the Step 2 notebook reuse it. Recovered words keep their real IAM IDs, so the **Aachen writer-disjoint splits still apply**.

In [ ]:
if os.path.exists(OFFICIAL_WORDS_TXT):
    print("Using the official full words.txt; no recovery needed.")
elif os.path.exists(RECOVERED_WORDS_TXT):
    print("Reusing", RECOVERED_WORDS_TXT)
else:
    !python scripts/recover_labels.py --data-root {DATA_ROOT} --out "{RECOVERED_WORDS_TXT}" \
        --hf-cache /content/hf_iam --report "{OUTPUT_DIR}/label_recovery.json" \
        --preview "{OUTPUT_DIR}/label_recovery_preview.png"
    if os.path.exists(RECOVERED_WORDS_TXT):
        run_setup()          # re-link with the recovered labels
        from IPython.display import Image as _I, display
        display(_I(os.path.join(OUTPUT_DIR, "label_recovery_preview.png")))

## Run the tests
EMNIST reading, orientation and download fallbacks, the CNN and resumable training, the splitter and normalization, decoding, the IAM run (broken images skipped), the report files and the three-way comparison. Every line should say `passed`.

In [ ]:
!python -m pytest -q tests/test_emnist.py tests/test_report.py tests/test_metrics.py

## GPU check
Training the character CNN uses the GPU. On a CPU runtime it still works, just slower (switch with Runtime → Change runtime type).

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_NAME = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print("device:", DEVICE_NAME)
if device.type != "cuda":
    print("No GPU: training will take ~30+ min. Consider Runtime -> Change runtime type -> T4 GPU.")

## EMNIST (byclass)
**Why byclass:** it is the only EMNIST split that keeps **upper- and lower-case letters apart** (62 classes: 0-9, A-Z, a-z), which our case-sensitive CER needs. `balanced` (47 classes) merges 15 lower-case letters such as c/o/s/x into their capitals, so it could never output those lower-case letters.

**Download, no sign-in:** the original NIST files are fetched from a Hugging Face mirror (two are tried) and, if both fail, from NIST itself (562 MB). Every file is checked (gzip, header, number of images) before use. The four files (~200 MB) are cached in your project folder (`emnist/`), so later sessions skip the download.

**Transposed images:** EMNIST stores every image transposed (rows and columns swapped). The code checks this on the digit '1' (upright it is a vertical stroke) and fixes it; the preview below should show upright characters.

In [ ]:
import os, shutil, numpy as np, matplotlib.pyplot as plt
from src.baselines import emnist as E
EMNIST_DRIVE = os.path.join(PROJECT_DIR, "emnist")       # cache on Drive (~200 MB)
EMNIST_LOCAL = "/content/emnist"
os.makedirs(EMNIST_LOCAL, exist_ok=True)
if os.path.isdir(EMNIST_DRIVE):                          # copy the cached files to the fast local disk
    for f in E.FILES.values():
        if os.path.exists(os.path.join(EMNIST_DRIVE, f)) and not os.path.exists(os.path.join(EMNIST_LOCAL, f)):
            shutil.copy(os.path.join(EMNIST_DRIVE, f), EMNIST_LOCAL)
emnist = E.load_emnist(EMNIST_LOCAL)
if emnist["source"] != "cached":                         # first time: keep a copy on Drive
    os.makedirs(EMNIST_DRIVE, exist_ok=True)
    for f in E.FILES.values():
        shutil.copy(os.path.join(EMNIST_LOCAL, f), EMNIST_DRIVE)
    print("cached on Drive:", EMNIST_DRIVE)
fig, axes = plt.subplots(2, 12, figsize=(10, 2))
for ax, i in zip(axes.ravel(), np.random.default_rng(0).choice(len(emnist["train_y"]), 24, replace=False)):
    ax.imshow(emnist["train_x"][i], cmap="gray"); ax.set_title(E.CLASSES[emnist["train_y"][i]], fontsize=9); ax.axis("off")
plt.suptitle("EMNIST samples after the orientation fix (should be upright, white on black)", fontsize=9)
plt.tight_layout(); plt.show()

## Train the character CNN
A small VGG-style CNN (5 conv layers, ~0.45 M parameters) on 95% of EMNIST byclass training images (5% held out for validation), for `EPOCHS` epochs. The images stay on the GPU, so an epoch should take well under a minute on a T4. Training uses:
- light **augmentation** (slant, rotation, scale, shift, thinner/thicker strokes), because IAM letters are slanted and written with different pens;
- mildly **re-weighted classes** (byclass has ~10x more digits than some lower-case letters) and label smoothing.

Checkpoints (`last.pt` after every epoch, `best.pt` by validation accuracy) are saved to `runs/emnist_cnn/` on Drive. **If Colab disconnects, re-run: training resumes** from the last finished epoch. Changing `EPOCHS`/`TRAIN_LIMIT` starts a new run.

In [ ]:
EPOCHS = 6
TRAIN_LIMIT = None        # e.g. 200_000 for a quicker run; None = all ~663k training images
EMNIST_RUN = os.path.join(PROJECT_DIR, "runs", "emnist_cnn")
clf_model, clf_info = E.train(emnist, EMNIST_RUN, epochs=EPOCHS, train_limit=TRAIN_LIMIT, device=device)
t = clf_info["test"]
print(f"EMNIST test accuracy {t['accuracy']:.4f} | case-insensitive {t['accuracy_case_insensitive']:.4f} | "
      f"digits {t['accuracy_digits']:.3f} upper {t['accuracy_upper']:.3f} lower {t['accuracy_lower']:.3f}")

## Load the same IAM splits as the CRNN
Same `make_datasets()` as Steps 1-3 (same labels, filters, Aachen writer-independent splits, and the same broken-image list). The splitter reads each original PNG itself; an image that can't be decoded is skipped and listed, exactly like the other models.

In [ ]:
from src.training.data import load_data
from src.baselines import char_segment as cs, emnist_pipeline as EP
datasets, charset, stats = load_data(DATA_ROOT, splits=(), preload=False, check_images=False)
print({s: len(datasets[s]) for s in ("train", "val", "test")})
# IAM character prior: how often each of the 62 characters occurs in the IAM *training* labels,
# relative to how often the classifier saw it in EMNIST (used only if it helps on validation).
PRIOR = cs.char_prior([it.text for it in datasets["train"].items], emnist["train_y"])

## The letter splitter on a few validation words
Red boxes = crops sent to the classifier, blue = small marks labelled by the punctuation rules. Boxes are drawn on the **slant-corrected** word. Below each word: the normalized 28x28 crops.

**Characters EMNIST doesn't have** (punctuation): EMNIST only knows 0-9, A-Z, a-z. Small marks are labelled by size/position rules instead: `.` (dot on the baseline), `,` (taller mark at the baseline), `'` (mark above the letters), `"` (two marks), `-` (flat mark in the middle), and for one-mark word images also `:` `;` `!`. Other symbols (`# & ( ) * + / ?`) are never output, so they always count as errors. That is a real limitation of this baseline, and the report should say so.

In [ ]:
import random
from src.report.predict import original_image
def preview(items, params, alpha=0.0):
    fig, axes = plt.subplots(len(items), 2, figsize=(10, 1.3 * len(items)), gridspec_kw={"width_ratios": [1, 1.4]})
    for (a, b), it in zip(np.atleast_2d(axes), items):
        g = cs.load_gray(it.path)
        pieces, crops, info = cs.segment_word(g, params, return_info=True)
        lp = EP.classify([{"crops": crops}], clf_model, device)[0][0]
        a.imshow(cs.deslanted_gray(g, info["shear"]), cmap="gray")
        for q in pieces:
            a.add_patch(plt.Rectangle((q["x0"], q["y0"]), q["x1"] - q["x0"], q["y1"] - q["y0"], fill=False,
                                      ec="r" if q["kind"] == "char" else "b"))
        a.set_title(f"true {it.text!r} -> {cs.decode(pieces, lp, PRIOR, alpha)!r}", fontsize=8); a.axis("off")
        b.imshow(np.hstack(list(crops)) if len(crops) else np.zeros((28, 28)), cmap="gray"); b.axis("off")
    plt.tight_layout(); plt.show()
preview(random.Random(3).sample(datasets["val"].items, 4), cs.SegParams())

## Tune on validation words (never on test)
Two settings are chosen by **validation CER** on `N_TUNE` random validation words:
- the **width factor**: the expected character width as a multiple of the pen's stroke width (on IAM training words, width / characters / stroke width has a median of about 7-8); a blob wider than ~1.5 expected widths is cut;
- the **IAM character prior** on/off: the classifier's log-probabilities plus log(frequency in IAM *training* labels / frequency in EMNIST). It mostly fixes case and digit/letter confusions (`o` vs `O` vs `0`), because IAM text is mostly lower-case letters.

In [ ]:
N_TUNE = 2000
tune_items = random.Random(0).sample(datasets["val"].items, min(N_TUNE, len(datasets["val"])))
SEG_PARAMS, ALPHA, TUNING = EP.tune(tune_items, clf_model, device, PRIOR)
print(f"chosen: width factor {SEG_PARAMS.width_factor} x stroke width, IAM prior {'on' if ALPHA else 'off'} "
      f"(val CER {TUNING[0]['cer']:.4f} on {len(tune_items):,} words)")

## Settings and time estimate
`N_VAL` / `N_TEST` = number of words to run (`None` = the whole split). Use the full splits if you can: the comparison table always scores every model on the words they all have, so a subset also stays fair. The cell times 300 words first.

In [ ]:
N_VAL  = None     # e.g. 2000; None = all validation words
N_TEST = None     # e.g. 5000; None = all test words
probe = random.Random(1).sample(datasets["test"].items, min(300, len(datasets["test"])))
_, pinfo = EP.run(probe, clf_model, device, SEG_PARAMS, PRIOR, ALPHA, progress=False)
ms = pinfo["ms_per_word"]
print(f"{ms:.1f} ms/word (segmentation {pinfo['seg_ms_per_word']:.1f} ms on the CPU + classification)")
for split in ("val", "test"):
    print(f"  {split}: all {len(datasets[split]):,} words ~{len(datasets[split]) * ms / 60000:.1f} min")
def pick(split, n):
    items = datasets[split].items
    return items if not n or n >= len(items) else random.Random(0).sample(items, n)

## Run on val and test
Predictions are saved to `outputs/report/emnist_<split>_predictions.csv` (word ID, writer, truth, prediction, number of pieces).

In [ ]:
preds, infos, full_sizes = {}, {}, {}
for split, n in (("val", N_VAL), ("test", N_TEST)):
    items = pick(split, n)
    full_sizes[split] = len(datasets[split])
    print(f"{split}: {len(items):,} of {full_sizes[split]:,} words")
    preds[split], infos[split] = EP.run(items, clf_model, device, SEG_PARAMS, PRIOR, ALPHA)
    EP.save_rows(preds[split], os.path.join(REPORT_DIR, f"emnist_{split}_predictions.csv"))
    s = EP.scores(preds[split])
    print(f"  {split}: CER {s['cer']:.4f} | WER {s['wer']:.4f} | case-insensitive CER {s['cer_case_insensitive']:.4f} "
          f"| {infos[split]['ms_per_word']:.1f} ms/word")

# Report outputs
Saved to `outputs/report/` (same folder as Steps 2-3): figures as PNG (300 dpi) + PDF, tables as CSV + LaTeX booktabs `.tex`.

In [ ]:
from IPython.display import Image, Markdown, display
from src.report.manifest import Report, future_work
from src.report import emnist_report as er, crnn_report as cr, compare
from src.report.tables import csv_to_markdown
rep = Report(REPORT_DIR, step=4)
def show(paths):
    for p in paths or []:
        if p.endswith(".png"):
            display(Image(p, width=700))
show_table = lambda name: display(Markdown(csv_to_markdown(os.path.join(REPORT_DIR, name + ".csv"))))

### The character classifier on its own
`table_emnist_classifier`: EMNIST test accuracy (overall, case-insensitive, digits / upper / lower), size and training time.

In [ ]:
er.save_classifier(rep, clf_info)
show_table("table_emnist_classifier")

### Baseline metrics and tuning
`table_emnist_metrics`: CER/WER on val and test (case-sensitive, like the other models), case-insensitive CER, time per word. `table_emnist_tuning`: the validation grid (chosen row in bold).

In [ ]:
settings = {**cs.params_dict(SEG_PARAMS), "iam_prior": bool(ALPHA), "n_tune": len(tune_items)}
enums = er.save_metrics(rep, preds, infos, full_sizes, TUNING, settings, DEVICE_NAME, clf_info)
show_table("table_emnist_metrics"); show_table("table_emnist_tuning")
print("CER on words with only letters/digits (test):", enums["test"]["cer_letters_digits_only_words"])

### Segmentation examples
`fig_emnist_segmentation`: 4 randomly chosen test words (not cherry-picked) with the character boxes, the 28x28 crops and the predicted characters.

In [ ]:
ex = er.segmentation_examples(datasets["test"], infos["test"]["segs"], infos["test"]["logps"], PRIOR, ALPHA, n=4, seed=0)
show(er.save_segmentation_figure(rep, ex))

### Qualitative: typical failures
`fig_emnist_failures`: 12 randomly chosen **wrong** predictions (red), at original resolution.

In [ ]:
show(er.save_failure_grid(rep, datasets["test"], preds["test"], n=12, seed=0))

### Error analysis
Same analysis as for the CRNN and Tesseract: top character substitutions, most common wrong words, CER by word length.

In [ ]:
_, ebd, epaths = cr.save_error_analysis(rep, preds["test"], system="EMNIST baseline", prefix="emnist")
print("test errors:", ebd)
show(epaths[:2]); show_table("table_emnist_word_errors")
show(cr.save_cer_by_length(rep, preds["test"], system="EMNIST baseline", prefix="emnist")[:2])

### Baseline pipeline diagram
`fig_emnist_pipeline`.

In [ ]:
show(er.save_pipeline(rep))

### CRNN vs Tesseract vs EMNIST baseline
`table_comparison` (+ `fig_cer_by_length_comparison`): every model with results, on **the same words** (matched by word ID). Needs Step 2's results (and Step 3's for Tesseract) in `outputs/report/`; whichever notebook runs last rebuilds it with all of them.

In [ ]:
if compare.save_comparison(rep):
    show_table("table_comparison")
    show([os.path.join(REPORT_DIR, "fig_cer_by_length_comparison.png")])

### Summary and list of files

In [ ]:
summary = rep.write_summary(future_work(rep))
display(Markdown(open(summary).read()))

In [ ]:
print("Saved report files:")
rep.list_files();

## Notes for the report
- The EMNIST classifier is good at **isolated** characters (its own test accuracy is in `table_emnist_classifier`), but on IAM words the errors are dominated by **segmentation**: cursive letters are joined, so cuts fall inside letters or miss boundaries, and every wrong cut costs insertions/deletions. That gap is the motivation for a segmentation-free CRNN + CTC model.
- **Case:** the main CER/WER are case-sensitive, exactly like the CRNN and Tesseract. EMNIST can't tell some upper/lower-case pairs apart (o/O, s/S, c/C...), so the case-insensitive CER is reported as an extra.
- **Punctuation:** handled by position/size rules for `. , ' " - : ; !`; the other symbols can't be produced (EMNIST has no punctuation).
- **Tuning was done on validation words only**; the test set was run once with the chosen settings.
- Neither baseline was trained on IAM; the CRNN was.